# DQN Project Walkthrough

In [2]:
from pathlib import Path
import csv
import json
import sys

import numpy as np
import torch
from IPython.display import Image, display

repo_root = Path.cwd()
if not (repo_root / "dqn_project").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
from dqn_project.agent import DQNAgent
from dqn_project.configs import DQNConfig, get_config
from dqn_project.train import run_training
from dqn_project.utils import evaluate_agent, get_env_dimensions, make_env, set_seed

In [ ]:
set_seed(42)

demo_config = get_config("CartPole-v1").with_overrides(
    seed=42,
    device="auto",
    max_episodes=1000,
    eval_every_episodes=5,
    eval_episodes=3,
    batch_size=32,
    min_buffer_size=32,
    buffer_size=5000,
    epsilon_decay_steps=2000,
)
demo_config.validate()
demo_config.to_dict()


In [ ]:
env = make_env(env_id=demo_config.env_id, seed=demo_config.seed)
obs_dim, num_actions = get_env_dimensions(env)
agent = DQNAgent(observation_dim=obs_dim, num_actions=num_actions, config=demo_config)

print(f"Environment: {demo_config.env_id}")
print(f"Observation dim: {obs_dim}")
print(f"Num actions: {num_actions}")
print(f"Device: {agent.device}")
print("\nQ-network:")
print(agent.q_network)

In [ ]:
for step in [0, 100, 500, 1000, 2000, 5000]:
    agent.total_env_steps = step
    print(f"step={step:4d} -> epsilon={agent.epsilon:.4f}")

agent.total_env_steps = 0

In [ ]:
# Collect transitions and run train_step() manually.
observation, _ = env.reset(seed=demo_config.seed)
losses = []

for t in range(120):
    action = agent.select_action(observation)
    next_observation, reward, terminated, truncated, _ = env.step(action)

    agent.store(
        observation=observation,
        action=action,
        reward=reward,
        next_observation=next_observation,
        terminated=terminated,
        truncated=truncated,
    )

    loss = agent.train_step()
    if loss is not None:
        losses.append(loss)

    if t < 5:
        print(
            f"t={t:02d} action={action} reward={reward:.1f} terminated={terminated} truncated={truncated} buffer={len(agent.replay_buffer)}"
        )

    observation = next_observation
    if terminated or truncated:
        observation, _ = env.reset()

print(f"\nBuffer size: {len(agent.replay_buffer)}")
print(f"Gradient steps: {agent.total_gradient_steps}")
if losses:
    print(f"Last loss: {losses[-1]:.6f}")
else:
    print("No training step yet (buffer may be too small).")


In [ ]:
batch = agent.replay_buffer.sample(batch_size=8)

print("Batch shapes:")
print("observations   :", batch.observations.shape)
print("actions        :", batch.actions.shape)
print("rewards        :", batch.rewards.shape)
print("next_observations:", batch.next_observations.shape)
print("terminateds    :", batch.terminateds.shape)
print("truncateds     :", batch.truncateds.shape)

print("\nFirst sampled action/reward:")
print(batch.actions[0], batch.rewards[0])


In [ ]:
env.close()
print("Closed demo environment.")

## Run a Short End-to-End Training Job

This uses the project training loop and writes artifacts to `dqn_project/checkpoints/...`.


In [ ]:
train_config = get_config("CartPole-v1").with_overrides(
    seed=123,
    device="auto",
    max_episodes=1000,
    eval_every_episodes=5,
    eval_episodes=2,
    batch_size=64,
    min_buffer_size=64,
    buffer_size=5000,
    epsilon_decay_steps=1500,
    run_name= "text-v2"
)
train_config.validate()

run_dir = run_training(train_config)
print("Run dir:", run_dir)


In [ ]:
run_dir = Path(run_dir)
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
summary


In [ ]:
# Show generated training plots inline.
for name in ["reward_curve.png", "epsilon_curve.png", "loss_curve.png", "eval_reward_curve.png"]:
    plot_path = run_dir / name
    if plot_path.exists():
        print(name)
        display(Image(filename=str(plot_path)))

In [ ]:
# Load best checkpoint and run greedy evaluation manually.
best_checkpoint = Path(summary["best_checkpoint"])
checkpoint = torch.load(best_checkpoint, map_location="cpu")
ckpt_config = DQNConfig.from_dict(checkpoint["config"]).with_overrides(device="auto")
ckpt_config.validate()

eval_env = make_env(env_id=ckpt_config.env_id, seed=999)
obs_dim, num_actions = get_env_dimensions(eval_env)
eval_env.close()

eval_agent = DQNAgent(observation_dim=obs_dim, num_actions=num_actions, config=ckpt_config)
eval_agent.load(best_checkpoint)

eval_stats = evaluate_agent(
    agent=eval_agent,
    env_id=ckpt_config.env_id,
    num_episodes=5,
    seed=999,
    render=False,
)

rewards = [s.reward for s in eval_stats]
lengths = [s.length for s in eval_stats]
print("Rewards per episode:", rewards)
print("Mean reward:", float(np.mean(rewards)))
print("Mean length:", float(np.mean(lengths)))


In [ ]:
run_dir = Path("/home/prasanna/coding/classic-RL/dqn_project/checkpoints")
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
summary


## Custom Environment Playground
 

In [4]:
from dqn_project.configs import supported_env_ids

In [5]:

print(supported_env_ids())

for env_id in ["SimpleCartPole-v0", "SimpleLunarLander-v0"]:
    env = make_env(env_id=env_id, seed=7)
    observation, info = env.reset(seed=7)
    obs_dim, num_actions = get_env_dimensions(env)
    print(f"\n{env_id}")
    print("observation_dim:", obs_dim)
    print("num_actions:", num_actions)
    print("initial observation:", observation)
    env.close()

['CartPole-v1', 'LunarLander-v3', 'SimpleCartPole-v0', 'SimpleLunarLander-v0']

SimpleCartPole-v0
observation_dim: 4
num_actions: 2
initial observation: [ 0.01250955  0.03972138  0.02756857 -0.02747928]

SimpleLunarLander-v0
observation_dim: 8
num_actions: 4
initial observation: [ 0.06254774  1.0794427   0.01654114 -0.02324378 -0.0319734   0.01494214
  0.          0.        ]


### Step Through One Environment

Change `env_id` and the `actions` list to see how the state changes. This is the fastest way to understand what your DQN sees.

In [ ]:
env_id = "SimpleCartPole-v0"
# env_id = "SimpleLunarLander-v0"

env = make_env(env_id=env_id, seed=11, render_mode="ansi")
observation, info = env.reset(seed=11)
print("reset observation:", observation)
print("reset info:", info)

actions = [0, 1, 1, 0, 1, 0, 0, 1]
for t, action in enumerate(actions, start=1):
    next_observation, reward, terminated, truncated, info = env.step(action)
    print(f"step={t:02d} action={action} reward={reward:+.3f}")
    print("  next_observation:", next_observation)
    print("  terminated/truncated:", terminated, truncated, "info:", info)
    if terminated or truncated:
        break

env.close()

### Random Policy Rollout

This runs random actions for one episode. Try changing the env id, seed, or `max_steps`.

In [ ]:
env_id = "SimpleLunarLander-v0"
max_steps = 200

env = make_env(env_id=env_id, seed=21)
observation, info = env.reset(seed=21)
total_reward = 0.0

for step in range(1, max_steps + 1):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if step <= 10 or terminated or truncated:
        print(
            f"step={step:03d} action={action} reward={reward:+.3f} "
            f"total={total_reward:+.3f} done={terminated or truncated}"
        )
        print("  obs:", observation)
        print("  info:", info)
    if terminated or truncated:
        break

print("episode length:", step)
print("total reward:", total_reward)
env.close()

### Train DQN On A Custom Preset

This uses the same `run_training` function as CartPole/LunarLander, but points it at the local environment id. Keep the episode count small while experimenting.

In [ ]:
custom_config = get_config("SimpleCartPole-v0").with_overrides(
    seed=123,
    device="auto",
    max_episodes=50,
    eval_every_episodes=10,
    eval_episodes=2,
    batch_size=32,
    min_buffer_size=64,
    buffer_size=5000,
    epsilon_decay_steps=1500,
    run_name="notebook-simple-cartpole",
)
custom_config.validate()
custom_config.to_dict()

In [ ]:
custom_run_dir = run_training(custom_config)
custom_run_dir

### Evaluate A Trained Checkpoint Live

For the text-style custom renderers, `render=True` prints state values each step. For Gymnasium's real envs, render may open a viewer depending on your local setup.

In [ ]:
custom_summary = json.loads((Path(custom_run_dir) / "summary.json").read_text(encoding="utf-8"))
custom_checkpoint = Path(custom_summary["best_checkpoint"])
checkpoint = torch.load(custom_checkpoint, map_location="cpu")
ckpt_config = DQNConfig.from_dict(checkpoint["config"]).with_overrides(device="auto")

eval_env = make_env(env_id=ckpt_config.env_id, seed=999)
obs_dim, num_actions = get_env_dimensions(eval_env)
eval_env.close()

custom_agent = DQNAgent(observation_dim=obs_dim, num_actions=num_actions, config=ckpt_config)
custom_agent.load(custom_checkpoint)

evaluate_agent(
    agent=custom_agent,
    env_id=ckpt_config.env_id,
    num_episodes=2,
    seed=999,
    render=True,
)